# 10eecheck: Earth Engine check for Stage 4 (2021 floods)

This notebook only checks that Earth Engine works in the KL2 environment and that the images we planned to use are really there. It changes no files.

## Section 0: settings

In [1]:
EE_PROJECT = "uhisea"          # <- your Google Cloud project for Earth Engine
GRID_FILE = "data/processed/hex_grid.parquet"   # <- used only to get the Klang Valley box
BOX_MARGIN_DEG = 0.02          # <- extra space around the hexagons

FLOOD_FROM = "2021-12-19"      # <- first day to look for flood images
FLOOD_TO = "2021-12-24"        # <- day after the last day (end is exclusive)
BEFORE_FROM = "2021-11-10"     # <- first day of the dry comparison period
BEFORE_TO = "2021-12-13"       # <- day after the last day of the dry period

## Step 1: connect to Earth Engine

The first time, a browser window opens to log in to Google. After that the login is remembered on this computer.

In [2]:
import ee

try:
    ee.Initialize(project=EE_PROJECT)
    print("Connected to Earth Engine, project:", EE_PROJECT)
except Exception as err:
    print("Not logged in yet (" + type(err).__name__ + "). Starting login...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("Connected to Earth Engine, project:", EE_PROJECT)

Connected to Earth Engine, project: uhisea


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## Step 2: the Klang Valley box

Taken from the Stage 1 hexagon grid, so Stage 4 uses the same area.

In [3]:
import pandas as pd

grid = pd.read_parquet(GRID_FILE)
west, east = grid["lon"].min() - BOX_MARGIN_DEG, grid["lon"].max() + BOX_MARGIN_DEG
south, north = grid["lat"].min() - BOX_MARGIN_DEG, grid["lat"].max() + BOX_MARGIN_DEG
region = ee.Geometry.Rectangle([west, south, east, north])
print(f"{len(grid):,} hexagons; box west {west:.3f}, south {south:.3f}, east {east:.3f}, north {north:.3f}")

4,003 hexagons; box west 101.196, south 2.581, east 101.986, north 3.419


## Step 3: Sentinel-1 radar images

We expect two flood images (20 and 22 December 2021) and three or four earlier dry-weather images for each. Times are shown in Malaysia time.

In [6]:
from datetime import datetime, timezone, timedelta
def list_s1(start, end):
    col = (ee.ImageCollection("COPERNICUS/S1_GRD")
           .filterBounds(region)
           .filterDate(start, end)
           .filter(ee.Filter.eq("instrumentMode", "IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    info = col.toList(200).map(lambda im: ee.Feature(None, {
        "id": ee.Image(im).id(),
        "time": ee.Image(im).get("system:time_start"),
        "orbit": ee.Image(im).get("relativeOrbitNumber_start"),
        "pass": ee.Image(im).get("orbitProperties_pass"),
        "sat": ee.Image(im).get("platform_number"),
    })).getInfo()
    rows = []
    for f in info:
        p = f["properties"]
        t = datetime.fromtimestamp(p["time"] / 1000, tz=timezone.utc) + timedelta(hours=8)
        rows.append({"malaysia_time": t.strftime("%Y-%m-%d %H:%M"), "orbit": p["orbit"], "pass": p["pass"], "satellite": p["sat"], "id": p["id"]})
    return pd.DataFrame(rows)

print("Flood period:")
display(list_s1(FLOOD_FROM, FLOOD_TO))
print("Dry period:")
display(list_s1(BEFORE_FROM, BEFORE_TO))

Flood period:


,malaysia_time,orbit,pass,satellite,id
0,2021-12-20 19:34,70,ASCENDING,A,S1A_IW_GRDH_1SDV_20211220T113408_20211220T1134...
1,2021-12-20 19:34,70,ASCENDING,A,S1A_IW_GRDH_1SDV_20211220T113433_20211220T1134...
2,2021-12-22 06:56,91,DESCENDING,A,S1A_IW_GRDH_1SDV_20211221T225607_20211221T2256...


Dry period:


,malaysia_time,orbit,pass,satellite,id
0,2021-11-14 19:34,70,ASCENDING,A,S1A_IW_GRDH_1SDV_20211114T113410_20211114T1134...
1,2021-11-14 19:34,70,ASCENDING,A,S1A_IW_GRDH_1SDV_20211114T113435_20211114T1135...
2,2021-11-16 06:56,91,DESCENDING,A,S1A_IW_GRDH_1SDV_20211115T225608_20211115T2256...
3,2021-11-21 07:04,164,DESCENDING,A,S1A_IW_GRDH_1SDV_20211120T230402_20211120T2304...
4,2021-11-21 07:04,164,DESCENDING,A,S1A_IW_GRDH_1SDV_20211120T230427_20211120T2304...
5,2021-11-21 19:26,172,ASCENDING,A,S1A_IW_GRDH_1SDV_20211121T112600_20211121T1126...
6,2021-11-21 19:26,172,ASCENDING,A,S1A_IW_GRDH_1SDV_20211121T112625_20211121T1126...
7,2021-11-26 19:34,70,ASCENDING,A,S1A_IW_GRDH_1SDV_20211126T113409_20211126T1134...
8,2021-11-26 19:34,70,ASCENDING,A,S1A_IW_GRDH_1SDV_20211126T113434_20211126T1134...
9,2021-11-28 06:56,91,DESCENDING,A,S1A_IW_GRDH_1SDV_20211127T225608_20211127T2256...


## Step 4: helper datasets

We plan to use these to remove permanent water and steep or high ground, where radar floods are false alarms.

In [7]:
checks = {
    "JRC Global Surface Water": "JRC/GSW1_4/GlobalSurfaceWater",
    "MERIT Hydro (height above nearest drainage)": "MERIT/Hydro/v1_0_1",
    "Copernicus DEM 30 m": "COPERNICUS/DEM/GLO30",
}
for name, asset in checks.items():
    try:
        if asset.endswith("GLO30"):
            n = ee.ImageCollection(asset).filterBounds(region).size().getInfo()
            print(f"OK   {name}: {n} tiles over the box")
        else:
            bands = ee.Image(asset).bandNames().getInfo()
            print(f"OK   {name}: bands {bands}")
    except Exception as err:
        print(f"FAIL {name}: {type(err).__name__}: {str(err)[:150]}")

OK   JRC Global Surface Water: bands ['occurrence', 'change_abs', 'change_norm', 'seasonality', 'recurrence', 'transition', 'max_extent']
OK   MERIT Hydro (height above nearest drainage): bands ['elv', 'dir', 'wth', 'wat', 'upa', 'upg', 'hnd', 'viswth']


e:\IT Training\PORTFOLIO\KL2\.venv\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/DEM/GLO30! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/DEM/GLO30_2024_1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30

  warnings.warn(warning, category=DeprecationWarning)


OK   Copernicus DEM 30 m: 2 tiles over the box


___
**END**